# E3 · Dataset003_E3Face / Colab training

**Use this revised Dataset003 notebook, not the superseded E2 sampling-only version.**

Data: `handoff_e3.zip`, 106 T1/label pairs: **90 training entries from 67 people + 16 validation people**.
23 IXI training cases have `w2` copies, so **standard uniform sampling already implements IXI weight 2**.
There is no additional weighted sampler. Training label **6 is ignore**, not another anatomical class.
Validation labels remain 0–5 and the same 16 E2 subjects. No test/calibration data is included.

Same E2 environment, architecture, seed 42, 100 epochs, batch 2, four augmentation workers,
checkpoint every epoch. E3 starts from random weights. **All training switches start off.**

1. Finish uploading ZIP + SHA to `My Drive/TMS_E3/`.
2. Upload this notebook to Colab, choose NVIDIA L4/A100 and run setup/audit/preview.
3. Acknowledge the data, preprocess, complete the five-epoch pilot, then enable the main run.
4. Results: `My Drive/TMS_E3/runs/E3_Dataset003_001/`.

This E3 changes both annotation-ignore policy and training-source exposure. Clinical or facial-surface
improvement has not been established. Colab setup/idle time also consumes GPU credits.


In [ ]:
# Ayarlar. Devam ederken aynı deney kimliğini koru.
from pathlib import Path
import json, os, re, shutil, subprocess, sys, time

ARCHIVE = Path("/content/drive/MyDrive/TMS_E3/handoff_e3.zip")
ARCHIVE_SHA256_FILE = Path(str(ARCHIVE) + ".sha256")
RESULTS_BASE = Path("/content/drive/MyDrive/TMS_E3/runs")
EXPERIMENT_ID = "E3_Dataset003_001"
CANONICALIZE_TO_RAS = True  # IXI PSR saklanıyor, MultiAxial RAS; nnU-Net döndürmez. Açıklama 3. bölümde.
ACKNOWLEDGE_E3_DATA = False  # Önizlemeyi ve veri notlarını inceledikten sonra True yap.
RUN_5_EPOCH_PILOT = False
RUN_100_EPOCH_PRELIMINARY = False
RESUME_PILOT = False
RESUME_PRELIMINARY = False
RELEASE_RUNTIME_WHEN_DONE = False  # True: 100 epoch ve tam doğrulama bitince GPU oturumunu bırak.
CU_PER_HOUR = 0.0  # Açık oturumun gösterilen birim/saat değeri; 0 = bilinmiyor.
AVAILABLE_CU = 0.0
AUGMENTATION_WORKERS = 4
assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", EXPERIMENT_ID)
assert 1 <= AUGMENTATION_WORKERS <= 4 and CU_PER_HOUR >= 0 and AVAILABLE_CU >= 0
WORK = Path("/content/e3_dataset003_work")
TOOLS = Path("/content/e3_dataset003_tools")
VENV = Path("/content/e3-dataset003-venv")
PYTHON = VENV / "bin/python"
DATA = WORK / "handoff"
RAS_RAW = WORK / "raw_ras"
PREPROCESSED = WORK / "nnUNet_preprocessed"
RESULTS = RESULTS_BASE / EXPERIMENT_ID

assert CANONICALIZE_TO_RAS, "Keep E2-compatible RAS preparation enabled."


## 1 · Drive connection

Wait for both uploads, then mount the account containing `TMS_E3/handoff_e3.zip` and `.sha256`.


In [ ]:
assert sys.platform == "linux", "Bu defteri Google Colab'ın Linux oturumunda aç."
from google.colab import drive
drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Kalıcı Drive bağlantısı kurulamadı."
assert ARCHIVE.is_file(), (f"Paket bulunamadı: {ARCHIVE}\n"
    "Berke'nin TMS klasöründeki handoff_e3.zip için Drive'ım/TMS_E3 altına kısayol ekle ya da yolu düzelt.")
WORK.mkdir(parents=True, exist_ok=True)
TOOLS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
assert shutil.disk_usage("/content").free > 40 * 1024**3, "Kurulum, veri ve ön işleme için en az 40 GiB boş yer gerekli."

assert ARCHIVE_SHA256_FILE.is_file(), "Wait for both ZIP and .sha256 to finish uploading."


## 2 · Pinned training environment

Same Python 3.11, nnU-Net 2.8.1 and torch 2.6.0+cu124 as E2. Embedded sources are reviewed code; archive contents are data only.


In [ ]:
# Reviewed embedded sources; no archive file is executed.
EMBEDDED = {'e2_helpers.py': '"""E2 Colab helpers: check K1\'s handoff, put every case in one axis order, build training commands.\n\nImporting this module never starts training. Functions that read NIfTI import nibabel lazily, so the\narchive checks also run in Colab\'s own kernel.\n"""\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\nDATASET = "Dataset002_E2Charm"\nDATASET_ID = "2"\nLABELS = {"background": 0, "skin": 1, "skull": 2, "csf": 3, "gray_matter": 4, "white_matter": 5}\nTRAINERS = ("nnUNetTrainer_E2Smoke", "nnUNetTrainer_E2Preliminary")\nMAX_ARCHIVE_BYTES = 20_000_000_000\nGRID_ATOL_MM = 1e-3\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _layout(names):\n    """Where dataset.json and splits_final.json sit inside the archive (a top folder is allowed)."""\n    marker = f"nnUNet_raw/{DATASET}/dataset.json"\n    found = [name for name in names if name == marker or name.endswith("/" + marker)]\n    require(len(found) == 1, f"Arşivde tam bir tane {marker} olmalı; bulunan: {len(found)}.")\n    prefix = found[0][: -len(marker)]\n    preferred = f"{prefix}nnUNet_preprocessed/{DATASET}/splits_final.json"\n    splits = [preferred] if preferred in names else [n for n in names if PurePosixPath(n).name == "splits_final.json"]\n    require(len(splits) == 1, f"Arşivde tek bir splits_final.json bulunmalı; bulunan: {len(splits)}.")\n    return {"dataset": f"{prefix}nnUNet_raw/{DATASET}", "raw": f"{prefix}nnUNet_raw", "splits": splits[0]}\n\n\ndef extract_handoff(archive_path, destination, expected_sha256):\n    """Extract K1\'s archive once, after its SHA-256 matches the published value; return its layout."""\n    archive_path, destination = Path(archive_path), Path(destination).resolve()\n    digest = sha256(archive_path)\n    require(digest == expected_sha256, f"Arşivin SHA-256 değeri yayımlanan pakete uymuyor: {digest}")\n    marker, layout_file = destination / ".archive_sha256", destination / ".layout.json"\n    if destination.exists():\n        require(marker.is_file() and marker.read_text().strip() == digest,\n                "Bu klasör başka bir arşivden açılmış. Yeni bir hedef klasör seç.")\n        layout = json.loads(layout_file.read_text())\n        return {key: destination / value for key, value in layout.items()}\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive_path) as archive:\n        infos = archive.infolist()\n        # Archives made by Windows PowerShell 5.1 store "\\" separators; read them as "/".\n        entries = [(info, info.filename.replace("\\\\", "/")) for info in infos]\n        files = [(info, name) for info, name in entries if not name.endswith("/")]\n        names = [name for _, name in files]\n        require(len(names) == len(set(names)), "Arşivde aynı yol iki kez geçiyor.")\n        require(sum(info.file_size for info in infos) <= MAX_ARCHIVE_BYTES, "Arşiv beklenenden çok büyük.")\n        for info, name in entries:\n            path = PurePosixPath(name)\n            require(not path.is_absolute() and ".." not in path.parts and ":" not in name,\n                    f"Güvensiz arşiv yolu: {info.filename}")\n            require(not stat.S_ISLNK(info.external_attr >> 16), f"Arşivde bağlantı (symlink) var: {info.filename}")\n        layout = _layout(names)\n        with tempfile.TemporaryDirectory(prefix="e2-extract-", dir=destination.parent) as folder:\n            temporary = Path(folder) / "content"\n            for info, name in files:\n                output = temporary / name\n                output.parent.mkdir(parents=True, exist_ok=True)\n                with archive.open(info) as source, output.open("xb") as target:\n                    shutil.copyfileobj(source, target, length=1024 * 1024)\n            (temporary / ".archive_sha256").write_text(digest + "\\n")\n            (temporary / ".layout.json").write_text(json.dumps(layout, indent=2) + "\\n")\n            temporary.rename(destination)\n    return {key: destination / value for key, value in layout.items()}\n\n\ndef _cases(dataset):\n    return sorted(p.name[: -len("_0000.nii.gz")] for p in (Path(dataset) / "imagesTr").glob("*_0000.nii.gz"))\n\n\ndef dataset_fingerprint(dataset, splits):\n    """One hash over every file name and content in the dataset folder plus the split."""\n    dataset = Path(dataset)\n    lines = [f"{sha256(path)}  {path.relative_to(dataset).as_posix()}"\n             for path in sorted(p for p in dataset.rglob("*") if p.is_file())]\n    lines.append(f"{sha256(splits)}  splits_final.json")\n    return hashlib.sha256("\\n".join(lines).encode()).hexdigest()\n\n\ndef audit(dataset, splits, contract):\n    """Technical checks against the contract; returns a report. Not an anatomical review."""\n    import nibabel as nib\n    import numpy as np\n\n    dataset, splits = Path(dataset), Path(splits)\n    for name in ("imagesTs", "labelsTs"):\n        require(not (dataset / name).exists(), f"Test klasörü eğitim paketine girmemeli: {name}")\n    metadata = json.loads((dataset / "dataset.json").read_text())\n    require(metadata.get("labels") == LABELS, "Etiket sözleşmesi 0–5 (deri, kafatası, BOS, GM, WM) değil.")\n    require(metadata.get("channel_names") == {"0": "T1"}, "Tek T1 kanalı bekleniyordu.")\n    require(metadata.get("file_ending") == ".nii.gz", ".nii.gz dosyaları bekleniyordu.")\n    folds = json.loads(splits.read_text())\n    require(len(folds) == 1 and set(folds[0]) == {"train", "val"}, "Tek bir sabit train/val bölmesi bekleniyordu.")\n    train, val = folds[0]["train"], folds[0]["val"]\n    require(sorted(train) == sorted(contract["train"]) and sorted(val) == sorted(contract["val"]),\n            "Bölme, K1\'in atamasından hesaplanan listeyle aynı değil.")\n    require(not set(train) & set(val), "Bir vaka hem eğitimde hem doğrulamada.")\n    cases = set(train) | set(val)\n    require(metadata.get("numTraining") == len(cases), "dataset.json numTraining bölmeyle uyuşmuyor.")\n    images = {p.name for p in (dataset / "imagesTr").iterdir()}\n    labels = {p.name for p in (dataset / "labelsTr").iterdir()}\n    require(images == {f"{case}_0000.nii.gz" for case in cases}, "imagesTr\'de eksik, fazla ya da yanlış adlı dosya var.")\n    require(labels == {f"{case}.nii.gz" for case in cases}, "labelsTr\'de eksik, fazla ya da yanlış adlı dosya var.")\n    report = {"dataset": DATASET, "train_count": len(train), "validation_count": len(val), "cases": [],\n              "axis_codes": {}, "warnings": []}\n    for case in sorted(cases):\n        image = nib.load(dataset / "imagesTr" / f"{case}_0000.nii.gz")\n        label = nib.load(dataset / "labelsTr" / f"{case}.nii.gz")\n        require(image.ndim == 3 and label.shape == image.shape, f"{case}: 3B görüntü ve aynı boyutta etiket bekleniyordu.")\n        require(np.allclose(image.affine, label.affine, rtol=0, atol=GRID_ATOL_MM), f"{case}: görüntü ve etiket aynı ızgarada değil.")\n        values = np.asanyarray(label.dataobj)\n        require(np.all(values == np.round(values)), f"{case}: etiketler tam sayı değil.")\n        present = sorted(int(v) for v in np.unique(values))\n        require(set(present) <= set(LABELS.values()), f"{case}: 0–5 dışında etiket var: {present}")\n        require(np.isfinite(np.asanyarray(image.dataobj)).all(), f"{case}: görüntüde sonlu olmayan değer var.")\n        codes = "".join(nib.aff2axcodes(image.affine))\n        report["axis_codes"][codes] = report["axis_codes"].get(codes, 0) + 1\n        report["cases"].append({"case": case, "split": "train" if case in train else "val", "shape": list(image.shape),\n                                "axis_codes": codes, "units": image.header.get_xyzt_units()[0],\n                                "zooms": [round(float(z), 4) for z in image.header.get_zooms()[:3]],\n                                "labels_present": present})\n    if len(report["axis_codes"]) > 1:\n        report["warnings"].append(f"Vakalar farklı eksen sıralarında saklanmış: {report[\'axis_codes\']}. "\n                                  "nnU-Net görüntüyü döndürmez.")\n    report["dataset_sha256"] = dataset_fingerprint(dataset, splits)\n    report["split_sha256"] = sha256(splits)\n    return report\n\n\ndef _to_ras(path, out_path):\n    """Permute and flip axes toward RAS without interpolation; returns the source axis codes."""\n    import nibabel as nib\n    from nibabel import orientations\n    import numpy as np\n\n    image = nib.load(path)\n    data = np.asanyarray(image.dataobj)\n    transform = orientations.io_orientation(image.affine)\n    moved = orientations.apply_orientation(data, transform).copy()\n    affine = image.affine @ orientations.inv_ornt_aff(transform, data.shape)\n    result = nib.Nifti1Image(moved, affine)\n    result.set_data_dtype(data.dtype)\n    result.set_sform(affine, code=1)\n    result.set_qform(affine, code=1)\n    result.header.set_xyzt_units(image.header.get_xyzt_units()[0])\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    nib.save(result, str(out_path))\n    return "".join(nib.aff2axcodes(image.affine)), transform.tolist()\n\n\ndef write_ras_dataset(dataset, raw_out):\n    """Copy the dataset into raw_out with every image and label in RAS order; returns the change log.\n\n    Voxel values are never interpolated: each case is an exact axis permutation and flip of the original,\n    and the world position of every voxel is unchanged."""\n    dataset, target = Path(dataset), Path(raw_out) / DATASET\n    require(not target.exists(), f"{target} zaten var; yeni bir klasör seç.")\n    partial = Path(raw_out) / f".partial_{DATASET}"\n    if partial.exists():  # left by an interrupted run; never used for training\n        shutil.rmtree(partial)\n    changes = {}\n    for case in _cases(dataset):\n        image_codes, transform = _to_ras(dataset / "imagesTr" / f"{case}_0000.nii.gz",\n                                         partial / "imagesTr" / f"{case}_0000.nii.gz")\n        label_codes, label_transform = _to_ras(dataset / "labelsTr" / f"{case}.nii.gz", partial / "labelsTr" / f"{case}.nii.gz")\n        require(image_codes == label_codes and transform == label_transform, f"{case}: görüntü ve etiket aynı dönüşümü almadı.")\n        changes[case] = {"source_axis_codes": image_codes, "to_ras": transform}\n    shutil.copyfile(dataset / "dataset.json", partial / "dataset.json")\n    partial.rename(target)\n    return changes\n\n\ndef install_split(splits, preprocessed):\n    target = Path(preprocessed) / DATASET / "splits_final.json"\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if target.exists():\n        require(json.loads(target.read_text()) == json.loads(Path(splits).read_text()),\n                "Kayıtlı bölme farklı; üzerine yazılmıyor.")\n    else:\n        shutil.copyfile(splits, target)\n    return sha256(target)\n\n\ndef training_command(executable, trainer, fold_directory, resume=False):\n    require(trainer in TRAINERS, "Beklenmeyen E2 eğiticisi.")\n    folder = Path(fold_directory)\n    if resume:\n        checkpoints = [folder / f"checkpoint_{name}.pth" for name in ("final", "latest", "best")]\n        checkpoint = next((path for path in checkpoints if path.is_file()), None)\n        require(checkpoint is not None, "Devam istendi ama checkpoint yok; sessizce baştan başlatılmıyor.")\n        require(not (folder / "validation/summary.json").is_file(),\n                "Bu çalışmanın doğrulaması bitmiş; sonuçlara bak ya da yeni deney kimliği kullan.")\n        require(zipfile.is_zipfile(checkpoint), "Checkpoint eksik ya da beklenmeyen biçimde.")\n        with zipfile.ZipFile(checkpoint) as archive:\n            require(archive.testzip() is None, "Checkpoint bütünlük kontrolünden geçmedi.")\n    else:\n        require(not folder.exists(), "Çalışma klasörü var. Devam seçeneğini ya da yeni deney kimliğini kullan.")\n    return [str(executable), DATASET_ID, "3d_fullres", "0", "-tr", trainer, "-device", "cuda"] + (["--c"] if resume else [])\n\n\ndef validation_complete(fold_directory, expected_cases):\n    """True only when the final 3d_fullres validation covered every validation case."""\n    summary = Path(fold_directory) / "validation" / "summary.json"\n    if not summary.is_file():\n        return False\n    metrics = json.loads(summary.read_text())\n    return len(metrics.get("metric_per_case", [])) == expected_cases\n\n\ndef experiment_id_ok(value):\n    return re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", value) is not None\n\n\ndef preview(dataset, cases, destination):\n    """Mid-plane overlays drawn in voxel order, so differently stored cases look different here."""\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    from matplotlib.colors import ListedColormap\n    import nibabel as nib\n    import numpy as np\n\n    colors = ListedColormap([(0, 0, 0, 0), (.99, .65, .52, .45), (1, .94, .70, .5), (.4, .64, 1, .5),\n                             (.65, .32, .8, .5), (.9, .9, .9, .5)])\n    figure, axes = plt.subplots(len(cases), 3, figsize=(12, 3.6 * len(cases)), squeeze=False)\n    for row, case in enumerate(cases):\n        image = nib.load(Path(dataset) / "imagesTr" / f"{case}_0000.nii.gz")\n        data = np.asanyarray(image.dataobj).astype(np.float32)\n        label = np.asanyarray(nib.load(Path(dataset) / "labelsTr" / f"{case}.nii.gz").dataobj)\n        low, high = np.percentile(data[data > 0], [1, 99]) if (data > 0).any() else (0, 1)\n        for axis in range(3):\n            index = [slice(None)] * 3\n            index[axis] = data.shape[axis] // 2\n            ax = axes[row, axis]\n            ax.imshow(np.rot90(data[tuple(index)]), cmap="gray", vmin=low, vmax=high)\n            ax.imshow(np.rot90(label[tuple(index)]), cmap=colors, vmin=0, vmax=5, interpolation="nearest")\n            ax.set_title(f"{case} · {\'\'.join(nib.aff2axcodes(image.affine))} · eksen {axis}", fontsize=9)\n            ax.axis("off")\n    figure.tight_layout()\n    figure.savefig(destination, dpi=90)\n    plt.close(figure)\n\n\ndef main():\n    import argparse\n    parser = argparse.ArgumentParser(description="E2 handoff checks for the Colab notebook.")\n    commands = parser.add_subparsers(dest="command", required=True)\n    check = commands.add_parser("audit")\n    check.add_argument("dataset", type=Path)\n    check.add_argument("splits", type=Path)\n    check.add_argument("contract", type=Path)\n    check.add_argument("out", type=Path)\n    ras = commands.add_parser("ras")\n    ras.add_argument("dataset", type=Path)\n    ras.add_argument("raw_out", type=Path)\n    ras.add_argument("out", type=Path)\n    view = commands.add_parser("preview")\n    view.add_argument("dataset", type=Path)\n    view.add_argument("out", type=Path)\n    view.add_argument("cases", nargs="+")\n    args = parser.parse_args()\n    if args.command == "audit":\n        report = audit(args.dataset, args.splits, json.loads(args.contract.read_text()))\n        args.out.write_text(json.dumps(report, indent=2) + "\\n")\n        print(f"{report[\'train_count\']} eğitim / {report[\'validation_count\']} doğrulama; eksen sıraları: {report[\'axis_codes\']}")\n    elif args.command == "ras":\n        changes = write_ras_dataset(args.dataset, args.raw_out)\n        args.out.write_text(json.dumps(changes, indent=2) + "\\n")\n        moved = sorted(case for case, change in changes.items() if change["source_axis_codes"] != "RAS")\n        print(f"RAS\'a çevrilen vaka: {len(moved)} / {len(changes)}")\n    else:\n        preview(args.dataset, args.cases, args.out)\n        print(args.out)\n\n\nif __name__ == "__main__":\n    main()\n', 'colab_helpers.py': '"""Small reusable Colab helpers; importing this module never starts training."""\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport signal\nimport stat\nimport subprocess\nimport tempfile\nimport time\nimport zipfile\n\nDATASET = "Dataset001_MultiAxial"\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef gpu_probe():\n    import importlib.metadata\n    import torch\n    require(importlib.metadata.version("nnunetv2") == "2.8.1", "Expected nnU-Net 2.8.1.")\n    require(torch.__version__.split("+")[0] == "2.6.0" and torch.version.cuda == "12.4", "Expected torch 2.6.0 / CUDA 12.4.")\n    require(torch.cuda.is_available(), "Choose a GPU runtime in Colab; CUDA is unavailable.")\n    props = torch.cuda.get_device_properties(0)\n    require(props.total_memory >= 12 * 1024**3, "This handoff requests a GPU with at least 12 GB VRAM.")\n    layer = torch.nn.Conv3d(1, 2, 3).cuda()\n    value = torch.randn(1, 1, 12, 12, 12, device="cuda")\n    with torch.autocast("cuda"):\n        result = layer(value).square().mean()\n    result.backward()\n    torch.cuda.synchronize()\n    del layer, value, result\n    torch.cuda.empty_cache()\n    return {"gpu": props.name, "vram_gib": props.total_memory / 1024**3,\n            "torch": torch.__version__, "cuda": torch.version.cuda,\n            "nnunetv2": importlib.metadata.version("nnunetv2"), "conv3d_backward_passed": True}\n\n\ndef pack_handoff(root, report, destination):\n    root, destination = Path(root).resolve(), Path(destination).resolve()\n    require(not destination.exists(), "Archive already exists; choose a new output filename.")\n    require(not any((parent / ".git").exists() for parent in destination.parents), "Keep MRI archives outside Git.")\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    manifest = {"files": report["files"], "dataset_sha256": report["dataset_sha256"]}\n    partial = destination.with_suffix(destination.suffix + ".partial")\n    try:\n        with zipfile.ZipFile(partial, "x", compression=zipfile.ZIP_STORED) as archive:\n            for relative, digest in sorted(manifest["files"].items()):\n                source = root / relative\n                require(sha256(source) == digest, f"Input changed after audit: {relative}")\n                archive.write(source, relative)\n            archive.writestr("expected_files.json", json.dumps(manifest, indent=2) + "\\n")\n        partial.rename(destination)\n    except BaseException:\n        partial.unlink(missing_ok=True)\n        raise\n    return sha256(destination)\n\n\ndef extract_handoff(archive_path, destination):\n    archive_path, destination = Path(archive_path), Path(destination).resolve()\n    archive_digest = sha256(archive_path)\n    marker = destination / ".archive_sha256"\n    if destination.exists():\n        require(marker.is_file() and marker.read_text().strip() == archive_digest,\n                "Existing extraction belongs to another archive. Choose a fresh destination.")\n        return destination\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive_path) as archive:\n        names = archive.namelist()\n        require(len(names) == len(set(names)), "Archive has duplicate paths.")\n        require("expected_files.json" in names and archive.getinfo("expected_files.json").file_size < 2_000_000,\n                "Use the prepared handoff_k2_e1.zip with its checksum manifest.")\n        manifest = json.loads(archive.read("expected_files.json"))\n        expected = manifest["files"]\n        require(set(names) == set(expected) | {"expected_files.json"}, "Archive contains unexpected or missing entries.")\n        require(sum(info.file_size for info in archive.infolist()) <= 20_000_000_000, "Archive is too large.")\n        for info in archive.infolist():\n            path = PurePosixPath(info.filename)\n            require(not path.is_absolute() and ".." not in path.parts and "\\\\" not in info.filename,\n                    "Unsafe archive path.")\n            require(not stat.S_ISLNK(info.external_attr >> 16) and not info.is_dir(), "Archive links/directories are not supported.")\n            allowed = (info.filename in ("splits_final.json", "expected_files.json", f"nnUNet_raw/{DATASET}/dataset.json")\n                       or (path.parent.as_posix() in (f"nnUNet_raw/{DATASET}/imagesTr", f"nnUNet_raw/{DATASET}/labelsTr")\n                           and path.name.endswith(".nii.gz")))\n            require(allowed, "Archive contains files outside the E1 dataset.")\n        with tempfile.TemporaryDirectory(prefix="e1-extract-", dir=destination.parent) as folder:\n            temporary = Path(folder)\n            for info in archive.infolist():\n                output = temporary / info.filename\n                output.parent.mkdir(parents=True, exist_ok=True)\n                with archive.open(info) as source, output.open("xb") as target:\n                    shutil.copyfileobj(source, target, length=1024 * 1024)\n                if info.filename in expected:\n                    require(sha256(output) == expected[info.filename], f"Archive checksum mismatch: {info.filename}")\n            (temporary / ".archive_sha256").write_text(archive_digest + "\\n")\n            temporary.rename(destination)\n    return destination\n\n\ndef verify_manifest(root, report):\n    manifest = json.loads((Path(root) / "expected_files.json").read_text())\n    require(report["files"] == manifest["files"] and report["dataset_sha256"] == manifest["dataset_sha256"],\n            "Extracted data differs from the audited handoff manifest.")\n\n\ndef install_split(root, preprocessed):\n    source = Path(root) / "splits_final.json"\n    target = Path(preprocessed) / DATASET / "splits_final.json"\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if target.exists():\n        require(json.loads(target.read_text()) == json.loads(source.read_text()), "Existing split differs; refusing to overwrite it.")\n    else:\n        shutil.copyfile(source, target)\n    return sha256(target)\n\n\ndef training_command(executable, trainer, fold_directory, resume=False):\n    require(trainer in ("nnUNetTrainer_E1Smoke", "nnUNetTrainer_E1Preliminary"), "Unexpected E1 trainer.")\n    folder = Path(fold_directory)\n    if resume:\n        checkpoints = [folder / f"checkpoint_{name}.pth" for name in ("final", "latest", "best")]\n        checkpoint = next((path for path in checkpoints if path.is_file()), None)\n        require(checkpoint is not None, "Resume requested but no checkpoint exists; refusing to silently restart.")\n        require(not (folder / "validation/summary.json").is_file(), "Run already has completed validation; inspect results or use a new experiment ID.")\n        require(zipfile.is_zipfile(checkpoint), "Checkpoint is incomplete or uses an unexpected format.")\n        with zipfile.ZipFile(checkpoint) as archive:\n            require(archive.testzip() is None, "Checkpoint archive failed its integrity check.")\n    else:\n        require(not folder.exists(), "Run directory exists. Use resume or a new experiment ID; no overwrite allowed.")\n    return [str(executable), "1", "3d_fullres", "0", "-tr", trainer, "-device", "cuda"] + (["--c"] if resume else [])\n\n\ndef record_run(path, contract):\n    path = Path(path)\n    if path.exists():\n        require(json.loads(path.read_text()) == contract, "Experiment contract changed; use a new experiment ID.")\n    else:\n        path.parent.mkdir(parents=True, exist_ok=True)\n        path.write_text(json.dumps(contract, indent=2) + "\\n")\n\n\ndef run_logged(command, log_path, env=None):\n    """Tail a durable log; stop owned process group when the notebook is interrupted."""\n    log_path = Path(log_path)\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    started = time.monotonic()\n    child_env = dict(os.environ if env is None else env)\n    child_env["PYTHONUNBUFFERED"] = "1"\n    with log_path.open("x", buffering=1) as output:\n        process = subprocess.Popen(command, stdout=output, stderr=subprocess.STDOUT,\n                                   stdin=subprocess.DEVNULL, env=child_env, start_new_session=True)\n        try:\n            with log_path.open(errors="replace") as reader:\n                while True:\n                    text = reader.read()\n                    if text:\n                        print(text, end="", flush=True)\n                    code = process.poll()\n                    if code is not None:\n                        print(reader.read(), end="", flush=True)\n                        if code != 0:\n                            raise subprocess.CalledProcessError(code, command)\n                        return {"elapsed_seconds": time.monotonic() - started, "exit_code": code}\n                    time.sleep(.3)\n        finally:\n            # Colab is Linux; group ownership prevents orphaned augmentation workers.\n            try:\n                os.killpg(process.pid, signal.SIGTERM)\n            except ProcessLookupError:\n                pass\n            if process.poll() is None:\n                try:\n                    process.wait(timeout=10)\n                except subprocess.TimeoutExpired:\n                    os.killpg(process.pid, signal.SIGKILL)\n                    process.wait()\n            try:\n                os.killpg(process.pid, signal.SIGKILL)\n            except ProcessLookupError:\n                pass\n', 'requirements-colab.lock.txt': '# This file was autogenerated by uv via the following command:\n#    uv pip compile tools/e1_training/requirements-colab.txt --python-version 3.11 --python-platform x86_64-unknown-linux-gnu --torch-backend cu124 --output-file /private/tmp/e1-linux-requirements.txt\nacvl-utils==0.2.7\n    # via nnunetv2\nannotated-types==0.8.0\n    # via pydantic\nanyio==4.16.0\n    # via\n    #   httpx\n    #   httpx2\nbatchgenerators==0.25.6\n    # via\n    #   acvl-utils\n    #   batchgeneratorsv2\n    #   nnunetv2\nbatchgeneratorsv2==0.3.7\n    # via nnunetv2\nblosc2==4.14.1\n    # via\n    #   acvl-utils\n    #   nnunetv2\ncertifi==2026.10.23\n    # via\n    #   httpcore\n    #   httpx\n    #   requests\ncharset-normalizer==3.5.2\n    # via requests\nclick==8.5.0\n    # via huggingface-hub\ncloudpickle==3.1.2\n    # via joblib\nconnected-components-3d==4.1.1\n    # via acvl-utils\ncontourpy==1.3.3\n    # via matplotlib\ncycler==0.12.1\n    # via matplotlib\ndynamic-network-architectures==0.4.4\n    # via nnunetv2\neinops==0.8.2\n    # via\n    #   dynamic-network-architectures\n    #   nnunetv2\nfilelock==4.2.7\n    # via\n    #   huggingface-hub\n    #   torch\nfonttools==4.68.0\n    # via matplotlib\nfsspec==2026.9.0\n    # via\n    #   huggingface-hub\n    #   torch\ngraphviz==0.21\n    # via nnunetv2\nh11==0.16.0\n    # via\n    #   httpcore\n    #   httpcore2\nh2==4.4.1\n    # via httpx\nhf-xet==1.8.0\n    # via huggingface-hub\nhpack==4.2.0\n    # via h2\nhttpcore==1.0.9\n    # via httpx\nhttpcore2==2.16.0\n    # via httpx2\nhttpx==0.28.1\n    # via blosc2\nhttpx2==2.16.0\n    # via huggingface-hub\nhuggingface-hub==2.4.0\n    # via timm\nhyperframe==6.1.0\n    # via h2\nidna==3.20\n    # via\n    #   anyio\n    #   httpx\n    #   httpx2\n    #   requests\nimagecodecs==2026.1.14\n    # via nnunetv2\nimageio==2.38.1\n    # via scikit-image\nimportlib-resources==7.1.0\n    # via nibabel\njinja2==3.1.6\n    # via torch\njoblib==1.6.0\n    # via scikit-learn\nkiwisolver==1.5.1\n    # via matplotlib\nlazy-loader==0.6\n    # via scikit-image\nmarkdown-it-py==4.2.0\n    # via rich\nmarkupsafe==3.0.4\n    # via jinja2\nmatplotlib==3.11.2\n    # via\n    #   nnunetv2\n    #   seaborn\nmdurl==0.1.2\n    # via markdown-it-py\nmpmath==1.3.0\n    # via sympy\nmsgpack==1.2.4\n    # via blosc2\nnarwhals==2.28.0\n    # via scikit-learn\nndindex==1.10.1\n    # via blosc2\nnetworkx==3.6.1\n    # via\n    #   scikit-image\n    #   torch\nnibabel==5.4.2\n    # via\n    #   -r tools/e1_training/requirements-colab.txt\n    #   nnunetv2\nnnunetv2==2.8.1\n    # via -r tools/e1_training/requirements-colab.txt\nnumexpr==2.14.2\n    # via blosc2\nnumpy==1.26.4\n    # via\n    #   -r tools/e1_training/requirements-colab.txt\n    #   acvl-utils\n    #   batchgenerators\n    #   batchgeneratorsv2\n    #   blosc2\n    #   connected-components-3d\n    #   contourpy\n    #   dynamic-network-architectures\n    #   imagecodecs\n    #   imageio\n    #   matplotlib\n    #   nibabel\n    #   nnunetv2\n    #   numexpr\n    #   pandas\n    #   scikit-image\n    #   scikit-learn\n    #   scipy\n    #   seaborn\n    #   tifffile\n    #   torchvision\nnvidia-cublas-cu12==12.4.5.8\n    # via\n    #   nvidia-cudnn-cu12\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cuda-cupti-cu12==12.4.127\n    # via torch\nnvidia-cuda-nvrtc-cu12==12.4.127\n    # via torch\nnvidia-cuda-runtime-cu12==12.4.127\n    # via torch\nnvidia-cudnn-cu12==9.1.0.70\n    # via torch\nnvidia-cufft-cu12==11.2.1.3\n    # via torch\nnvidia-curand-cu12==10.3.5.147\n    # via torch\nnvidia-cusolver-cu12==11.6.1.9\n    # via torch\nnvidia-cusparse-cu12==12.3.1.170\n    # via\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cusparselt-cu12==0.6.2\n    # via torch\nnvidia-nccl-cu12==2.21.5\n    # via torch\nnvidia-nvjitlink-cu12==12.4.127\n    # via\n    #   nvidia-cusolver-cu12\n    #   nvidia-cusparse-cu12\n    #   torch\nnvidia-nvtx-cu12==12.4.127\n    # via torch\npackaging==26.3\n    # via\n    #   huggingface-hub\n    #   lazy-loader\n    #   matplotlib\n    #   nibabel\n    #   scikit-image\npandas==3.0.6\n    # via\n    #   batchgenerators\n    #   nnunetv2\n    #   seaborn\npillow==13.0.0\n    # via\n    #   batchgenerators\n    #   imageio\n    #   matplotlib\n    #   scikit-image\n    #   torchvision\npydantic==2.14.2\n    # via blosc2\npydantic-core==2.50.2\n    # via pydantic\npygments==2.21.0\n    # via rich\npyparsing==3.3.3\n    # via matplotlib\npython-dateutil==4.0.0\n    # via\n    #   matplotlib\n    #   pandas\npyyaml==6.0.3\n    # via\n    #   huggingface-hub\n    #   timm\n    #   yacs\nrequests==2.34.2\n    # via nnunetv2\nrich==15.0.0\n    # via blosc2\nsafetensors==0.8.0\n    # via timm\nscikit-image==0.26.0\n    # via\n    #   acvl-utils\n    #   batchgenerators\n    #   nnunetv2\nscikit-learn==1.9.1\n    # via\n    #   batchgenerators\n    #   nnunetv2\nscipy==1.17.1\n    # via\n    #   batchgenerators\n    #   nnunetv2\n    #   scikit-image\n    #   scikit-learn\nseaborn==0.13.2\n    # via nnunetv2\nsimpleitk==2.5.7\n    # via\n    #   acvl-utils\n    #   nnunetv2\nsympy==1.13.1\n    # via torch\nthreadpoolctl==3.7.0\n    # via\n    #   batchgenerators\n    #   blosc2\n    #   scikit-learn\ntifffile==2026.3.3\n    # via\n    #   nnunetv2\n    #   scikit-image\ntimm==1.0.22\n    # via dynamic-network-architectures\ntorch==2.6.0+cu124\n    # via\n    #   -r tools/e1_training/requirements-colab.txt\n    #   acvl-utils\n    #   batchgeneratorsv2\n    #   dynamic-network-architectures\n    #   nnunetv2\n    #   timm\n    #   torchvision\ntorchvision==0.21.0+cu124\n    # via timm\ntqdm==4.70.1\n    # via\n    #   acvl-utils\n    #   huggingface-hub\n    #   nnunetv2\ntriton==3.2.0\n    # via torch\ntruststore==0.10.5\n    # via\n    #   httpcore2\n    #   httpx2\ntyping-extensions==4.16.0\n    # via\n    #   anyio\n    #   httpx2\n    #   huggingface-hub\n    #   nibabel\n    #   pydantic\n    #   pydantic-core\n    #   torch\n    #   typing-inspection\ntyping-inspection==0.4.4\n    # via pydantic\ntzdata==2026.6\n    # via python-dateutil\nurllib3==2.8.0\n    # via requests\nyacs==0.1.8\n    # via nnunetv2\n', 'e3_data.py': '"""Dataset003 adapter around E2\'s reviewed archive/grid/RAS helpers.\n\nThe private module copy leaves E2 behavior unchanged. Archive contents are never code.\n"""\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nfrom pathlib import Path\n\nDATASET = "Dataset003_E3Face"\nDATASET_ID = "3"\nLABELS = {"background": 0, "skin": 1, "skull": 2, "csf": 3,\n          "gray_matter": 4, "white_matter": 5, "ignore": 6}\nHERE = Path(__file__).resolve().parent\nbase_path = HERE / "e2_helpers.py"\nif not base_path.is_file():\n    base_path = HERE.parent / "e2_training/e2_helpers.py"\nspec = importlib.util.spec_from_file_location("_e3_base_helpers", base_path)\n_base = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(_base)\n_base.DATASET, _base.DATASET_ID, _base.LABELS = DATASET, DATASET_ID, LABELS\nrequire, sha256 = _base.require, _base.sha256\nextract_handoff, write_ras_dataset, install_split = _base.extract_handoff, _base.write_ras_dataset, _base.install_split\ndataset_fingerprint = _base.dataset_fingerprint\n\n\ndef verify_sidecar(path, expected):\n    fields = Path(path).read_text(encoding="utf-8-sig").split()\n    require(fields and fields[0].lower() == expected, "Archive SHA-256 sidecar differs from the reviewed E3 package.")\n\n\ndef validate_split(train, val, contract):\n    require(len(train) == len(set(train)) and len(val) == len(set(val)), "Duplicate split entries.")\n    require(sorted(train) == sorted(contract["train"]) and sorted(val) == sorted(contract["val"]), "Fixed E3 split changed.")\n    copies = contract["copy_sources"]\n    require(all(c == source + "w2" and source.startswith("IXI") for c, source in copies.items()), "Unexpected copy identifiers.")\n    require(set(train) == set(contract["source_train"]) | set(copies), "Training entries differ from source people plus IXI copies.")\n    require(set(copies.values()) == {c for c in contract["source_train"] if c.startswith("IXI")}, "Every IXI training case needs exactly one copy.")\n    require(not set(contract["source_train"]) & set(val) and not set(train) & set(val), "Source-person training/validation leakage.")\n\n\ndef outside_acquisition(image):\n    """Zero voxels connected to a volume border, with the preparer\'s 6-connectivity."""\n    import numpy as np\n    from scipy import ndimage\n    components, _ = ndimage.label(image == 0)\n    border = np.unique(np.concatenate([components[0].ravel(), components[-1].ravel(),\n        components[:, 0].ravel(), components[:, -1].ravel(), components[:, :, 0].ravel(), components[:, :, -1].ravel()]))\n    return np.isin(components, border[border > 0])\n\n\ndef array_grid_hash(values, image):\n    """Decoded values/grid, not platform-specific gzip encoding."""\n    import numpy as np\n    h = hashlib.sha256()\n    h.update(json.dumps({"shape": list(values.shape), "dtype": values.dtype.str,\n                         "units": image.header.get_xyzt_units()[0]}, sort_keys=True).encode())\n    h.update(np.asarray(image.affine, dtype="<f8").tobytes())\n    h.update(values.tobytes(order="C"))\n    return h.hexdigest()\n\n\ndef audit(dataset, splits, contract):\n    import nibabel as nib\n    import numpy as np\n    dataset, splits = Path(dataset), Path(splits)\n    split = json.loads(splits.read_text(encoding="utf-8"))[0]\n    validate_split(split["train"], split["val"], contract)\n    report = _base.audit(dataset, splits, contract)\n    manifest_path = splits.parent / "e3_manifest.csv"\n    if "manifest_sha256" in contract:\n        require(sha256(manifest_path) == contract["manifest_sha256"], "E3 manifest hash changed.")\n    with manifest_path.open(newline="", encoding="utf-8-sig") as f:\n        rows = list(csv.DictReader(f))\n    manifest = {r["case_id"]: r for r in rows}\n    require(len(manifest) == len(rows) == len(split["train"] + split["val"])\n            and set(manifest) == set(split["train"] + split["val"]), "Manifest cases differ from the split.")\n    copies = contract["copy_sources"]\n    semantic = {}\n    for case in split["train"] + split["val"]:\n        source = copies.get(case, case)\n        row = manifest[case]\n        is_train = case in split["train"]\n        require(row["source_case"] == source and row["split"] == ("train" if is_train else "val")\n                and row["source"] == ("IXI" if source.startswith("IXI") else "MultiAxial")\n                and int(row["weight_copy"]) == (2 if case in copies else 1), f"{case}: manifest identity mismatch.")\n        if case in copies:\n            for folder, ending in (("imagesTr", "_0000.nii.gz"), ("labelsTr", ".nii.gz")):\n                require(sha256(dataset / folder / (case + ending)) == sha256(dataset / folder / (source + ending)),\n                        f"{case}: weighted copy differs from its source.")\n            continue\n        image = nib.load(dataset / "imagesTr" / f"{case}_0000.nii.gz")\n        label = nib.load(dataset / "labelsTr" / f"{case}.nii.gz")\n        values = np.asanyarray(label.dataobj)\n        image_values = np.asanyarray(image.dataobj)\n        semantic[case] = [array_grid_hash(image_values, image), array_grid_hash(values, label)]\n        require(image.header.get_xyzt_units()[0] == "mm" and np.isfinite(image.affine).all()\n                and abs(np.linalg.det(image.affine[:3, :3])) > 1e-8, f"{case}: invalid physical geometry.")\n        if is_train:\n            mask = outside_acquisition(image_values)\n            require(np.array_equal(mask, values == 6), f"{case}: ignore mask differs from border-connected zero image voxels.")\n            ml = float(np.count_nonzero(mask)) * float(np.prod(label.header.get_zooms()[:3])) / 1000\n            require(abs(ml - float(row["ignore_ml"])) <= .11, f"{case}: ignored volume differs from manifest.")\n        else:\n            require(not np.any(values == 6), f"Validation {case} contains ignore voxels.")\n    for case, source in copies.items():semantic[case] = semantic[source]\n    semantic_payload = {"pairs": semantic, "split": split,\n                        "metadata": json.loads((dataset / "dataset.json").read_text())}\n    report["semantic_dataset_sha256"] = hashlib.sha256(json.dumps(semantic_payload, sort_keys=True).encode()).hexdigest()\n    report.update({"unique_training_subjects": len(contract["source_train"]), "weighted_copies": len(copies),\n                   "sampling": "uniform_over_entries", "ignore_label": 6})\n    return report\n\n\ndef preview(dataset, cases, destination):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    from matplotlib.colors import ListedColormap\n    import nibabel as nib\n    import numpy as np\n    colors = ListedColormap([(0, 0, 0, 0), (.99, .65, .52, .45), (1, .94, .70, .5),\n                             (.4, .64, 1, .5), (.65, .32, .8, .5), (.9, .9, .9, .5)])\n    fig, axes = plt.subplots(len(cases), 3, figsize=(12, 3.6 * len(cases)), squeeze=False)\n    for row, case in enumerate(cases):\n        a = np.asanyarray(nib.load(Path(dataset) / "imagesTr" / f"{case}_0000.nii.gz").dataobj)\n        b = np.asanyarray(nib.load(Path(dataset) / "labelsTr" / f"{case}.nii.gz").dataobj)\n        for axis in range(3):\n            index = [slice(None)] * 3;index[axis] = a.shape[axis] // 2\n            plane, labels = a[tuple(index)].T, b[tuple(index)].T\n            axes[row, axis].imshow(plane, cmap="gray", origin="lower")\n            axes[row, axis].imshow(np.ma.masked_where(labels == 6, labels), cmap=colors, vmin=0, vmax=5, origin="lower")\n            axes[row, axis].set_title(f"{case} / axis {axis}; ignore hidden")\n            axes[row, axis].axis("off")\n    fig.tight_layout();fig.savefig(destination, dpi=130);plt.close(fig)\n\n\ndef main():\n    import argparse\n    parser = argparse.ArgumentParser(description="Dataset003 E3 audit and lossless RAS preparation.")\n    commands = parser.add_subparsers(dest="command", required=True)\n    check = commands.add_parser("audit")\n    for name in ("dataset", "splits", "contract", "out"):check.add_argument(name, type=Path)\n    ras = commands.add_parser("ras")\n    for name in ("dataset", "raw_out", "out"):ras.add_argument(name, type=Path)\n    view = commands.add_parser("preview")\n    view.add_argument("dataset", type=Path);view.add_argument("out", type=Path);view.add_argument("cases", nargs="+")\n    args = parser.parse_args()\n    if args.command == "audit":\n        report = audit(args.dataset, args.splits, json.loads(args.contract.read_text()))\n        args.out.write_text(json.dumps(report, indent=2) + "\\n", encoding="utf-8")\n        print(f"{report[\'train_count\']} entries / {report[\'unique_training_subjects\']} people train; {report[\'validation_count\']} validation; {report[\'axis_codes\']}")\n    elif args.command == "ras":\n        changes = write_ras_dataset(args.dataset, args.raw_out)\n        args.out.write_text(json.dumps(changes, indent=2) + "\\n", encoding="utf-8")\n        print(f"RAS preparation complete: {len(changes)} entries.")\n    else:\n        preview(args.dataset, args.cases, args.out);print(args.out)\n\n\nif __name__ == "__main__":main()\n', 'e3_helpers.py': '"""Dataset003 E3 experiment guards; importing this never starts training."""\n\nimport json\nimport math\nfrom pathlib import Path\nimport zipfile\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\n\n\ndef training_command(executable, trainer, fold_directory, resume=False):\n    require(trainer in ("nnUNetTrainer_E3Smoke", "nnUNetTrainer_E3Preliminary"), "Expected an E3 trainer.")\n    folder = Path(fold_directory)\n    if resume:\n        checkpoint = next((folder / f"checkpoint_{name}.pth" for name in ("final", "latest", "best")\n                           if (folder / f"checkpoint_{name}.pth").is_file()), None)\n        require(checkpoint is not None, "No checkpoint to resume; refusing a silent fresh run.")\n        require(not (folder / "validation/summary.json").exists(), "Validation already exists; inspect it before resuming.")\n        require(zipfile.is_zipfile(checkpoint), "Checkpoint container is incomplete.")\n        with zipfile.ZipFile(checkpoint) as archive:\n            require(archive.testzip() is None, "Checkpoint CRC failed.")\n    else:\n        require(not folder.exists(), "Run folder exists; use resume or a new experiment ID.")\n    return [str(executable), "3", "3d_fullres", "0", "-tr", trainer, "-device", "cuda"] + (["--c"] if resume else [])\n\n\ndef validation_rows(summary, expected_cases):\n    rows = summary["metric_per_case"]\n    cases = []\n    for row in rows:\n        names = [str(row[key]).replace("\\\\", "/").rsplit("/", 1)[-1] for key in ("prediction_file", "reference_file")]\n        require(names[0] == names[1] and names[0].endswith(".nii.gz"), "Validation image identities differ.")\n        cases.append(names[0][:-7])\n    require(len(cases) == len(set(cases)) == len(expected_cases) and set(cases) == set(expected_cases),\n            "Final validation must cover exactly the fixed E2/E3 validation subjects.")\n    return list(zip(cases, rows))\n\n\ndef validation_complete(fold_directory, expected_cases):\n    path = Path(fold_directory) / "validation/summary.json"\n    if not path.is_file():\n        return False\n    try:\n        summary = json.loads(path.read_text(encoding="utf-8"))\n        validation_rows(summary, expected_cases)\n        return math.isfinite(summary["foreground_mean"]["Dice"])\n    except (ValueError, KeyError, TypeError):\n        return False\n\n\ndef subgroup_metrics(summary, expected_cases):\n    rows = validation_rows(summary, expected_cases)\n    result = {}\n    for group, is_ixi in (("IXI", True), ("MultiAxial", False)):\n        selected = [row for case, row in rows if case.startswith("IXI") == is_ixi]\n        require(selected, f"No {group} validation cases.")\n        means = {}\n        for label in range(1, 6):\n            scores = [row["metrics"][str(label)]["Dice"] for row in selected]\n            finite = [score for score in scores if isinstance(score, (float, int)) and math.isfinite(score)]\n            require(finite, f"No finite {group} Dice for label {label}.")\n            means[str(label)] = sum(finite) / len(finite)\n        result[group] = {"cases": len(selected), "per_label_dice": means,\n                         "mean_foreground_dice": sum(means.values()) / 5}\n    return result\n\n\n\n\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser()\n    parser.add_argument("summary", type=Path)\n    parser.add_argument("contract", type=Path)\n    parser.add_argument("output", type=Path)\n    args = parser.parse_args()\n    output = subgroup_metrics(json.loads(args.summary.read_text()), json.loads(args.contract.read_text())["val"])\n    args.output.write_text(json.dumps(output, indent=2) + "\\n", encoding="utf-8")\n    print(json.dumps(output, indent=2))\n', 'trainers_e3.py': '"""Dataset003 E3: standard nnU-Net sampling/loss, seed 42, checkpoint each epoch."""\n\nimport random\nimport numpy as np\nimport torch\nfrom nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import nnUNetTrainer_100epochs\n\n\nclass nnUNetTrainer_E3Preliminary(nnUNetTrainer_100epochs):\n    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):\n        if configuration != "3d_fullres" or str(fold) != "0" or dataset_json.get("labels", {}).get("ignore") != 6:\n            raise ValueError("E3 requires 3d_fullres, fixed fold 0 and ignore label 6.")\n        random.seed(42)\n        np.random.seed(42)\n        torch.manual_seed(42)\n        super().__init__(plans, configuration, fold, dataset_json, device)\n        self.save_every = 1\n\n\nclass nnUNetTrainer_E3Smoke(nnUNetTrainer_E3Preliminary):\n    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):\n        super().__init__(plans, configuration, fold, dataset_json, device)\n        self.num_epochs = 5\n', 'e3_contract.json': '{\n  "experiment": "E3",\n  "dataset": "Dataset003_E3Face",\n  "archive": "handoff_e3.zip",\n  "archive_sha256": "c12f5fb233d829d20bdcc112e412a1eb9fe17158a4a56377cec3e1951eba7698",\n  "labels": {\n    "background": 0,\n    "skin": 1,\n    "skull": 2,\n    "csf": 3,\n    "gray_matter": 4,\n    "white_matter": 5,\n    "ignore": 6\n  },\n  "train": [\n    "GU002",\n    "GU006",\n    "GU008",\n    "GU009",\n    "GU010",\n    "GU015",\n    "GU016",\n    "GU017",\n    "GU019",\n    "GU020",\n    "GU021",\n    "GU024",\n    "GU025",\n    "GU027",\n    "GU030",\n    "GU032",\n    "GU035",\n    "GU036",\n    "GU038",\n    "GU039",\n    "GU040",\n    "IXI028",\n    "IXI028w2",\n    "IXI037",\n    "IXI037w2",\n    "IXI045",\n    "IXI045w2",\n    "IXI054",\n    "IXI054w2",\n    "IXI062",\n    "IXI062w2",\n    "IXI070",\n    "IXI070w2",\n    "IXI078",\n    "IXI078w2",\n    "IXI087",\n    "IXI087w2",\n    "IXI095",\n    "IXI095w2",\n    "IXI103",\n    "IXI103w2",\n    "IXI185",\n    "IXI185w2",\n    "IXI195",\n    "IXI195w2",\n    "IXI212",\n    "IXI212w2",\n    "IXI222",\n    "IXI222w2",\n    "IXI239",\n    "IXI239w2",\n    "IXI249",\n    "IXI249w2",\n    "IXI329",\n    "IXI329w2",\n    "IXI377",\n    "IXI377w2",\n    "IXI385",\n    "IXI385w2",\n    "IXI393",\n    "IXI393w2",\n    "IXI409",\n    "IXI409w2",\n    "IXI417",\n    "IXI417w2",\n    "IXI442",\n    "IXI442w2",\n    "NC004",\n    "NC008",\n    "NC010",\n    "NC011",\n    "NC013",\n    "NC014",\n    "NC016",\n    "NC017",\n    "NC018",\n    "NC020",\n    "NC021",\n    "NC022",\n    "NC024",\n    "NC027",\n    "NC031",\n    "NC033",\n    "NYU002",\n    "NYU003",\n    "NYU004",\n    "NYU006",\n    "NYU007",\n    "NYU008",\n    "NYU010"\n  ],\n  "val": [\n    "GU011",\n    "GU012",\n    "GU018",\n    "GU033",\n    "GU041",\n    "IXI148",\n    "IXI167",\n    "IXI294",\n    "IXI320",\n    "IXI368",\n    "IXI426",\n    "NC012",\n    "NC019",\n    "NC023",\n    "NC029",\n    "NYU005"\n  ],\n  "source_train": [\n    "GU002",\n    "GU006",\n    "GU008",\n    "GU009",\n    "GU010",\n    "GU015",\n    "GU016",\n    "GU017",\n    "GU019",\n    "GU020",\n    "GU021",\n    "GU024",\n    "GU025",\n    "GU027",\n    "GU030",\n    "GU032",\n    "GU035",\n    "GU036",\n    "GU038",\n    "GU039",\n    "GU040",\n    "IXI028",\n    "IXI037",\n    "IXI045",\n    "IXI054",\n    "IXI062",\n    "IXI070",\n    "IXI078",\n    "IXI087",\n    "IXI095",\n    "IXI103",\n    "IXI185",\n    "IXI195",\n    "IXI212",\n    "IXI222",\n    "IXI239",\n    "IXI249",\n    "IXI329",\n    "IXI377",\n    "IXI385",\n    "IXI393",\n    "IXI409",\n    "IXI417",\n    "IXI442",\n    "NC004",\n    "NC008",\n    "NC010",\n    "NC011",\n    "NC013",\n    "NC014",\n    "NC016",\n    "NC017",\n    "NC018",\n    "NC020",\n    "NC021",\n    "NC022",\n    "NC024",\n    "NC027",\n    "NC031",\n    "NC033",\n    "NYU002",\n    "NYU003",\n    "NYU004",\n    "NYU006",\n    "NYU007",\n    "NYU008",\n    "NYU010"\n  ],\n  "copy_sources": {\n    "IXI028w2": "IXI028",\n    "IXI037w2": "IXI037",\n    "IXI045w2": "IXI045",\n    "IXI054w2": "IXI054",\n    "IXI062w2": "IXI062",\n    "IXI070w2": "IXI070",\n    "IXI078w2": "IXI078",\n    "IXI087w2": "IXI087",\n    "IXI095w2": "IXI095",\n    "IXI103w2": "IXI103",\n    "IXI185w2": "IXI185",\n    "IXI195w2": "IXI195",\n    "IXI212w2": "IXI212",\n    "IXI222w2": "IXI222",\n    "IXI239w2": "IXI239",\n    "IXI249w2": "IXI249",\n    "IXI329w2": "IXI329",\n    "IXI377w2": "IXI377",\n    "IXI385w2": "IXI385",\n    "IXI393w2": "IXI393",\n    "IXI409w2": "IXI409",\n    "IXI417w2": "IXI417",\n    "IXI442w2": "IXI442"\n  },\n  "sampling": "uniform_over_entries",\n  "initialization": "random_seed_42",\n  "epochs": 100,\n  "manifest_sha256": "7c6cf6d4019ce4830dcd16a1267d922bb745039857ed50f5848891d0aa14f3c5",\n  "split_sha256": "ef817e7e11b210d222e42453516180e7e939bc0f6302daf7a8f951fce1e20b87",\n  "source_dataset_sha256": "418ec564afdd7c094b4e68544cd90ca4b3d6508f64f363228f5e34d777b831db",\n  "plans_sha256": "1f844faf95bce163ace9b0d57992f48679026f0b6cc64a4f18a21c37fa2b2885",\n  "e2_plans_sha256": "472f3ec4e104c3c2b1b54567455103a89f73a25279a4c3054f6226943858b935",\n  "lock_sha256": "22b96a20a6c58e7b5573d163953d7dd0b18d71ef515b5d3f90574bfd62bdda32",\n  "ras_semantic_sha256": "784f7f41a30c6ea3473f2761cd0cc0c4564bd6375014106ada48dc7a71c1f28b"\n}\n', 'e3_plans.json': '{\n  "dataset_name": "Dataset003_E3Face",\n  "plans_name": "nnUNetPlans",\n  "original_median_spacing_after_transp": [\n    1.0,\n    1.0,\n    1.0\n  ],\n  "original_median_shape_after_transp": [\n    255,\n    255,\n    176\n  ],\n  "image_reader_writer": "SimpleITKIO",\n  "transpose_forward": [\n    0,\n    1,\n    2\n  ],\n  "transpose_backward": [\n    0,\n    1,\n    2\n  ],\n  "configurations": {\n    "2d": {\n      "data_identifier": "nnUNetPlans_2d",\n      "preprocessor_name": "DefaultPreprocessor",\n      "batch_size": 65,\n      "patch_size": [\n        256,\n        192\n      ],\n      "median_image_size_in_voxels": [\n        249.0,\n        180.0\n      ],\n      "spacing": [\n        1.0,\n        1.0\n      ],\n      "normalization_schemes": [\n        "ZScoreNormalization"\n      ],\n      "use_mask_for_norm": [\n        false\n      ],\n      "resampling_fn_data": "resample_data_or_seg_to_shape",\n      "resampling_fn_seg": "resample_data_or_seg_to_shape",\n      "resampling_fn_data_kwargs": {\n        "is_seg": false,\n        "order": 3,\n        "order_z": 0,\n        "force_separate_z": null\n      },\n      "resampling_fn_seg_kwargs": {\n        "is_seg": true,\n        "order": 1,\n        "order_z": 0,\n        "force_separate_z": null\n      },\n      "resampling_fn_probabilities": "resample_data_or_seg_to_shape",\n      "resampling_fn_probabilities_kwargs": {\n        "is_seg": false,\n        "order": 1,\n        "order_z": 0,\n        "force_separate_z": null\n      },\n      "architecture": {\n        "network_class_name": "dynamic_network_architectures.architectures.unet.PlainConvUNet",\n        "arch_kwargs": {\n          "n_stages": 6,\n          "features_per_stage": [\n            32,\n            64,\n            128,\n            256,\n            512,\n            512\n          ],\n          "conv_op": "torch.nn.modules.conv.Conv2d",\n          "kernel_sizes": [\n            [\n              3,\n              3\n            ],\n            [\n              3,\n              3\n            ],\n            [\n              3,\n              3\n            ],\n            [\n              3,\n              3\n            ],\n            [\n              3,\n              3\n            ],\n            [\n              3,\n              3\n            ]\n          ],\n          "strides": [\n            [\n              1,\n              1\n            ],\n            [\n              2,\n              2\n            ],\n            [\n              2,\n              2\n            ],\n            [\n              2,\n              2\n            ],\n            [\n              2,\n              2\n            ],\n            [\n              2,\n              2\n            ]\n          ],\n          "n_conv_per_stage": [\n            2,\n            2,\n            2,\n            2,\n            2,\n            2\n          ],\n          "n_conv_per_stage_decoder": [\n            2,\n            2,\n            2,\n            2,\n            2\n          ],\n          "conv_bias": true,\n          "norm_op": "torch.nn.modules.instancenorm.InstanceNorm2d",\n          "norm_op_kwargs": {\n            "eps": 1e-05,\n            "affine": true\n          },\n          "dropout_op": null,\n          "dropout_op_kwargs": null,\n          "nonlin": "torch.nn.LeakyReLU",\n          "nonlin_kwargs": {\n            "inplace": true\n          }\n        },\n        "_kw_requires_import": [\n          "conv_op",\n          "norm_op",\n          "dropout_op",\n          "nonlin"\n        ]\n      },\n      "batch_dice": true\n    },\n    "3d_fullres": {\n      "data_identifier": "nnUNetPlans_3d_fullres",\n      "preprocessor_name": "DefaultPreprocessor",\n      "batch_size": 2,\n      "patch_size": [\n        160,\n        128,\n        112\n      ],\n      "median_image_size_in_voxels": [\n        249.0,\n        249.0,\n        180.0\n      ],\n      "spacing": [\n        1.0,\n        1.0,\n        1.0\n      ],\n      "normalization_schemes": [\n        "ZScoreNormalization"\n      ],\n      "use_mask_for_norm": [\n        false\n      ],\n      "resampling_fn_data": "resample_data_or_seg_to_shape",\n      "resampling_fn_seg": "resample_data_or_seg_to_shape",\n      "resampling_fn_data_kwargs": {\n        "is_seg": false,\n        "order": 3,\n        "order_z": 0,\n        "force_separate_z": null\n      },\n      "resampling_fn_seg_kwargs": {\n        "is_seg": true,\n        "order": 1,\n        "order_z": 0,\n        "force_separate_z": null\n      },\n      "resampling_fn_probabilities": "resample_data_or_seg_to_shape",\n      "resampling_fn_probabilities_kwargs": {\n        "is_seg": false,\n        "order": 1,\n        "order_z": 0,\n        "force_separate_z": null\n      },\n      "architecture": {\n        "network_class_name": "dynamic_network_architectures.architectures.unet.PlainConvUNet",\n        "arch_kwargs": {\n          "n_stages": 6,\n          "features_per_stage": [\n            32,\n            64,\n            128,\n            256,\n            320,\n            320\n          ],\n          "conv_op": "torch.nn.modules.conv.Conv3d",\n          "kernel_sizes": [\n            [\n              3,\n              3,\n              3\n            ],\n            [\n              3,\n              3,\n              3\n            ],\n            [\n              3,\n              3,\n              3\n            ],\n            [\n              3,\n              3,\n              3\n            ],\n            [\n              3,\n              3,\n              3\n            ],\n            [\n              3,\n              3,\n              3\n            ]\n          ],\n          "strides": [\n            [\n              1,\n              1,\n              1\n            ],\n            [\n              2,\n              2,\n              2\n            ],\n            [\n              2,\n              2,\n              2\n            ],\n            [\n              2,\n              2,\n              2\n            ],\n            [\n              2,\n              2,\n              2\n            ],\n            [\n              2,\n              2,\n              1\n            ]\n          ],\n          "n_conv_per_stage": [\n            2,\n            2,\n            2,\n            2,\n            2,\n            2\n          ],\n          "n_conv_per_stage_decoder": [\n            2,\n            2,\n            2,\n            2,\n            2\n          ],\n          "conv_bias": true,\n          "norm_op": "torch.nn.modules.instancenorm.InstanceNorm3d",\n          "norm_op_kwargs": {\n            "eps": 1e-05,\n            "affine": true\n          },\n          "dropout_op": null,\n          "dropout_op_kwargs": null,\n          "nonlin": "torch.nn.LeakyReLU",\n          "nonlin_kwargs": {\n            "inplace": true\n          }\n        },\n        "_kw_requires_import": [\n          "conv_op",\n          "norm_op",\n          "dropout_op",\n          "nonlin"\n        ]\n      },\n      "batch_dice": false\n    }\n  },\n  "experiment_planner_used": "ExperimentPlanner",\n  "label_manager": "LabelManager",\n  "foreground_intensity_properties_per_channel": {\n    "0": {\n      "max": 13588.0,\n      "mean": 419.5888977050781,\n      "median": 295.0,\n      "min": -61.0,\n      "percentile_00_5": 14.0,\n      "percentile_99_5": 3708.0,\n      "std": 540.198486328125\n    }\n  }\n}\n'}
import hashlib
for name, source in EMBEDDED.items():
    (TOOLS / name).write_text(source, encoding="utf-8")
sys.path.insert(0, str(TOOLS))
for module_name in ("e3_data", "e3_helpers", "colab_helpers"):
    sys.modules.pop(module_name, None)  # Reload this notebook's embedded helper versions.
from e3_data import extract_handoff, install_split, require, sha256, verify_sidecar, DATASET, DATASET_ID
from e3_helpers import training_command, validation_complete, subgroup_metrics
from colab_helpers import record_run, run_logged
CONTRACT = json.loads(EMBEDDED["e3_contract.json"])
LOCK_SHA256 = sha256(TOOLS / "requirements-colab.lock.txt")
TRAINER_SHA256 = sha256(TOOLS / "trainers_e3.py")
CONTRACT_SHA256 = sha256(TOOLS / "e3_contract.json")
require(LOCK_SHA256 == CONTRACT["lock_sha256"], "Pinned dependency lock changed.")
verify_sidecar(ARCHIVE_SHA256_FILE, CONTRACT["archive_sha256"])
print("Dataset003: 90 train entries / 67 people; 16 validation; ignore=6.")
print("Uniform sampling: 23 IXI copies already implement weight 2. No extra sampler.")


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.11.8"], check=True)
uv = [sys.executable, "-m", "uv"]
if not PYTHON.is_file():
    subprocess.run(uv + ["venv", str(VENV), "--python", "3.11"], check=True)
subprocess.run(uv + ["pip", "install", "--python", str(PYTHON), "--torch-backend", "cu124",
                     "-r", str(TOOLS / "requirements-colab.lock.txt")], check=True)
subprocess.run(uv + ["pip", "check", "--python", str(PYTHON)], check=True)
trainer_path = Path(subprocess.check_output([str(PYTHON), "-c",
    "from pathlib import Path; import nnunetv2; print(Path(nnunetv2.__file__).parent / 'training/nnUNetTrainer/variants/training_length/nnUNetTrainer_E3.py')"], text=True).strip().splitlines()[-1])
if trainer_path.exists():
    require(trainer_path.read_text(encoding="utf-8") == EMBEDDED["trainers_e3.py"], "Installed trainer differs; use a fresh E3 environment.")
else:
    trainer_path.write_text(EMBEDDED["trainers_e3.py"], encoding="utf-8")
ENV = os.environ.copy()
ENV.update({"nnUNet_raw": str(RAS_RAW if CANONICALIZE_TO_RAS else DATA), "nnUNet_preprocessed": str(PREPROCESSED),
            "nnUNet_results": str(RESULTS), "nnUNet_n_proc_DA": str(AUGMENTATION_WORKERS),
            "nnUNet_compile": "false", "PYTHONHASHSEED": "42", "MPLBACKEND": "Agg",
            "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"})
require(subprocess.check_output([str(PYTHON), "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"], text=True).strip() == "3.11", "Python sürümü 3.11 olmalı.")
gpu = json.loads(subprocess.check_output([str(PYTHON), "-c",
    "import sys,json; sys.path.insert(0, '/content/e3_dataset003_tools'); from colab_helpers import gpu_probe; print(json.dumps(gpu_probe()))"], env=ENV, text=True))
print(json.dumps(gpu, indent=2))
session_id = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
(RESULTS / f"session_{session_id}.json").write_text(json.dumps({**gpu, "cu_per_hour": CU_PER_HOUR, "available_cu": AVAILABLE_CU}, indent=2))
(RESULTS / f"environment_{session_id}.txt").write_text(subprocess.check_output(uv + ["pip", "freeze", "--python", str(PYTHON)], text=True))


## 3 · Verify the actual E3 archive and prepare RAS

Checks cover checksum, source-person split isolation, integer labels, aligned grids, all copy identities and exact border-connected ignore masks. The ZIP is copied locally before extraction. Every image and label is losslessly reoriented to RAS.


In [ ]:
verify_sidecar(ARCHIVE_SHA256_FILE, CONTRACT["archive_sha256"])
local_archive = WORK / CONTRACT["archive"]
if not local_archive.is_file():
    partial = local_archive.with_suffix(".partial")
    print("Copying uploaded E3 archive to local Colab disk...", flush=True)
    shutil.copyfile(ARCHIVE, partial)
    require(sha256(partial) == CONTRACT["archive_sha256"], "Uploaded ZIP incomplete or different; checksum failed.")
    partial.replace(local_archive)
layout = extract_handoff(local_archive, DATA, CONTRACT["archive_sha256"])
contract_path = TOOLS / "e3_contract.json"
stamp = str(time.time_ns())
run_logged([str(PYTHON), str(TOOLS / "e3_data.py"), "audit", str(layout["dataset"]), str(layout["splits"]),
            str(contract_path), str(WORK / "audit_source.json")], RESULTS / "logs" / f"audit_source_{stamp}.log", ENV)
source_report = json.loads((WORK / "audit_source.json").read_text())
require(source_report["dataset_sha256"] == CONTRACT["source_dataset_sha256"], "Source E3 dataset fingerprint changed.")
require(source_report["split_sha256"] == CONTRACT["split_sha256"], "Fixed E3 split changed.")
shutil.copyfile(WORK / "audit_source.json", RESULTS / f"audit_source_{stamp}.json")
if not (RAS_RAW / DATASET).exists():
    run_logged([str(PYTHON), str(TOOLS / "e3_data.py"), "ras", str(layout["dataset"]), str(RAS_RAW),
                str(WORK / "orientation_changes.json")], RESULTS / "logs" / f"ras_{stamp}.log", ENV)
TRAIN_DATASET = RAS_RAW / DATASET
shutil.copyfile(WORK / "orientation_changes.json", RESULTS / f"orientation_changes_{stamp}.json")
run_logged([str(PYTHON), str(TOOLS / "e3_data.py"), "audit", str(TRAIN_DATASET), str(layout["splits"]),
            str(contract_path), str(WORK / "audit_train.json")], RESULTS / "logs" / f"audit_train_{stamp}.log", ENV)
report = json.loads((WORK / "audit_train.json").read_text())
require(report["semantic_dataset_sha256"] == CONTRACT["ras_semantic_sha256"], "Prepared RAS values/grid differ from reviewed E3 data.")
require(report["axis_codes"] == {"RAS": 106}, "Every E3 entry must be RAS.")
shutil.copyfile(WORK / "audit_train.json", RESULTS / f"audit_train_{stamp}.json")
print("All E3 entry/split/ignore/copy/grid checks passed:", report["axis_codes"])


In [ ]:
# Yalnız eğitim vakaları; voksel düzleminde önizleme. Hepsi aynı yönde görünmeli.
train_ids = CONTRACT["source_train"]
shown = [c for c in train_ids if c.startswith("IXI")][:2] + [c for c in train_ids if not c.startswith("IXI")][:2]
preview = RESULTS / "e3_label_review.png"
subprocess.run([str(PYTHON), str(TOOLS / "e3_data.py"), "preview", str(TRAIN_DATASET), str(preview), *shown], check=True, env=ENV)
from IPython.display import display, Image
display(Image(filename=str(preview)))


**Data acknowledgement.** Training-only preview hides ignore voxels. References are CHARM-based with the existing E2 mapping/lesion rules, not independent manual truth. Inspect the notes and preview, then run `ACKNOWLEDGE_E3_DATA = True`. This does not constitute anatomical acceptance.


## 4 · Preprocess with the E2 architecture and fixed E3 split

Review the preview and run `ACKNOWLEDGE_E3_DATA = True` first. The saved E2 plan is reused with only its dataset name changed to Dataset003: 1 mm, patch 160×128×112, batch 2. The provided 90/16 split is installed after preprocessing. Ignore 6 is excluded by standard nnU-Net loss; there are still six prediction channels including background.


In [ ]:
require(ACKNOWLEDGE_E3_DATA, "Review the source notes and preview, then set ACKNOWLEDGE_E3_DATA = True.")
frozen_plan = TOOLS / "e3_plans.json"
require(sha256(frozen_plan) == CONTRACT["plans_sha256"], "E3 plan hash changed.")
marker = PREPROCESSED / ".e3_dataset003_input_sha256"
preprocessing_key = hashlib.sha256((report["semantic_dataset_sha256"] + LOCK_SHA256 + sha256(frozen_plan)).encode()).hexdigest()
plan_path = PREPROCESSED / DATASET / "nnUNetPlans.json"
if marker.is_file():
    require(marker.read_text().strip() == preprocessing_key and sha256(plan_path) == CONTRACT["plans_sha256"], "Cached preprocessing differs.")
else:
    run_logged([str(VENV / "bin/nnUNetv2_extract_fingerprint"), "-d", DATASET_ID, "--verify_dataset_integrity", "-np", "1"],
               RESULTS / "logs" / f"fingerprint_{time.time_ns()}.log", ENV)
    plan_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(frozen_plan, plan_path)
    run_logged([str(VENV / "bin/nnUNetv2_preprocess"), "-d", DATASET_ID, "-c", "3d_fullres", "-np", "1"],
               RESULTS / "logs" / f"preprocess_{time.time_ns()}.log", ENV)
    marker.write_text(preprocessing_key + "\n")
split_hash = install_split(layout["splits"], PREPROCESSED)
require(split_hash == CONTRACT["split_sha256"], "Installed fixed split changed.")
plans = json.loads(plan_path.read_text())
print(json.dumps(plans["configurations"]["3d_fullres"], indent=2))
contract = {"experiment": "E3", "dataset": DATASET, "initialization": "random_seed_42",
    "archive_sha256": CONTRACT["archive_sha256"], "source_dataset_sha256": source_report["dataset_sha256"],
    "dataset_sha256": report["semantic_dataset_sha256"], "fingerprint_kind": "decoded_values_and_grid",
    "split_sha256": split_hash, "plans_sha256": sha256(plan_path),
    "lock_sha256": LOCK_SHA256, "trainer_sha256": TRAINER_SHA256, "e3_contract_sha256": CONTRACT_SHA256,
    "data_helpers_sha256": sha256(TOOLS / "e3_data.py"), "run_helpers_sha256": sha256(TOOLS / "e3_helpers.py"),
    "base_helpers_sha256": sha256(TOOLS / "e2_helpers.py"), "sampling": "uniform_over_entries", "ignore_label": 6,
    "train_entries": 90, "unique_train_subjects": 67, "validation_subjects": 16, "canonicalized_to_ras": True,
    "configuration": "3d_fullres", "fold": 0, "nnunetv2": "2.8.1", "torch": "2.6.0+cu124", "seed": 42,
    "augmentation_workers": AUGMENTATION_WORKERS, "torch_compile": False, "preliminary_only": True,
    "test_accessed": False, "inference_outside_acquisition_mask_required": True}
record_run(RESULTS / "experiment_contract.json", contract)
for path in (plan_path, PREPROCESSED / DATASET / "dataset_fingerprint.json", TRAIN_DATASET / "dataset.json", layout["splits"],
             layout["splits"].parent / "e3_manifest.csv", *[TOOLS / n for n in EMBEDDED]):
    target = RESULTS / path.name
    if target.exists():
        require(sha256(target) == sha256(path), f"Saved evidence changed: {path.name}; use a new experiment ID.")
    else:
        shutil.copyfile(path, target)


## 5 · Required five-epoch GPU pilot

Set `RUN_5_EPOCH_PILOT = True`, run the next cell and wait for all five epochs plus all 16 validation outputs. Check finite losses and a final checkpoint. This verifies the live CUDA/augmentation environment. Pilot weights do not initialize the main run.


In [ ]:
def train_stage(trainer, enabled, resume):
    if not enabled:
        print(f"{trainer}: kapalı; eğitim başlatılmadı.")
        return None
    require(ACKNOWLEDGE_E3_DATA and (RESULTS / "experiment_contract.json").is_file(), "Veri/ön işleme adımları tamamlanmalı.")
    require(os.path.ismount("/content/drive"), "Checkpoint hedefi kalıcı Drive bağlantısında olmalı.")
    require(ENV["nnUNet_results"] == str(RESULTS) and ENV["nnUNet_n_proc_DA"] == str(AUGMENTATION_WORKERS),
            "Ayarlar değişti; ortam/veri/ön işleme hücrelerini yeniden çalıştır.")
    if trainer == "nnUNetTrainer_E3Preliminary" and not resume:
        pilot = RESULTS / DATASET / "nnUNetTrainer_E3Smoke__nnUNetPlans__3d_fullres/fold_0"
        require(validation_complete(pilot, CONTRACT["val"]) and (pilot / "checkpoint_final.pth").is_file(),
                "Complete the E3 five-epoch pilot before the fresh 100-epoch run.")
    record_run(RESULTS / "experiment_contract.json", contract)
    folder = RESULTS / DATASET / f"{trainer}__nnUNetPlans__3d_fullres" / "fold_0"
    command = training_command(VENV / "bin/nnUNetv2_train", trainer, folder, resume)
    log = RESULTS / "logs" / f"{trainer}_{time.time_ns()}.log"
    result = run_logged(command, log, ENV)
    (log.with_suffix(".json")).write_text(json.dumps(result, indent=2))
    print(f"Bitti: {result['elapsed_seconds']/60:.1f} dakika. Log: {log}")
    return folder
train_stage("nnUNetTrainer_E3Smoke", RUN_5_EPOCH_PILOT, RESUME_PILOT)


## 6 · Pilot timing estimate

The median post-first-epoch time estimates the 100-epoch loop. Installation, preprocessing and final validation add time; GPU type and interruption overhead can change it.


In [ ]:
logs = sorted((RESULTS / "logs").glob("nnUNetTrainer_E3Smoke_*.log"))
epoch_seconds = []
for log in logs:
    epoch_seconds.extend(float(value) for value in re.findall(r"Epoch time:\s*([0-9.]+)", log.read_text()))
if len(epoch_seconds) >= 2:
    import statistics
    hours = statistics.median(epoch_seconds[1:]) * 100 / 3600
    print(f"100 epoch eğitim döngüsü yaklaşık {hours:.2f} saat; ek aşamalar dahil değil.")
    if CU_PER_HOUR > 0:
        print(f"Girilen oranla yaklaşık {hours * CU_PER_HOUR:.1f} işlem birimi + ek aşamalar.")
else:
    print("Tahmin için 5 epoch kontrolünden en az iki epoch süresi gerekli (adım atlandıysa normal).")


## 7 · Main 100-epoch E3 run

Set `RUN_5_EPOCH_PILOT = False`, `RUN_100_EPOCH_PRELIMINARY = True`, and run this cell after pilot completion. Optional `RELEASE_RUNTIME_WHEN_DONE = True` requests runtime release after verified success. A fresh run starts from random weights. For the same interrupted run, keep the experiment ID and set `RESUME_PRELIMINARY = True`.


In [ ]:
final_folder = train_stage("nnUNetTrainer_E3Preliminary", RUN_100_EPOCH_PRELIMINARY, RESUME_PRELIMINARY)
if final_folder is not None:
    checkpoint = final_folder / "checkpoint_final.pth"
    require(checkpoint.is_file() and checkpoint.stat().st_size > 0, "Final checkpoint missing; runtime left active.")
    require(validation_complete(final_folder, CONTRACT["val"]), "Final validation identities/metrics incomplete; runtime left active.")
    metrics = json.loads((final_folder / "validation/summary.json").read_text())
    groups = subgroup_metrics(metrics, CONTRACT["val"])
    (RESULTS / "validation_subgroups.json").write_text(json.dumps(groups, indent=2) + "\n")
    (RESULTS / "run_success.json").write_text(json.dumps({"experiment": "E3", "dataset": DATASET,
        "checkpoint": str(checkpoint), "validation_cases": 16, "foreground_mean": metrics["foreground_mean"],
        "runtime_release_requested": RELEASE_RUNTIME_WHEN_DONE}, indent=2) + "\n")
    print("Raw E3 validation saved:", metrics["foreground_mean"], groups, flush=True)
    if RELEASE_RUNTIME_WHEN_DONE:
        from google.colab import runtime
        runtime.unassign()


## 8 · Final validation and delivery

Use full-volume final Dice, not epoch pseudo-Dice. These are raw predictions against unchanged E2 validation labels. Compare the same TTA/preprocessing settings and report IXI/MultiAxial subgroups. Preserve the whole run folder, code, plans, logs and validation outputs. Future inference must clear boundary-connected zero-signal regions; that separate postprocessing is not applied to raw validation here. No independent face-surface accuracy is claimed.


In [ ]:
folder = RESULTS / DATASET / "nnUNetTrainer_E3Preliminary__nnUNetPlans__3d_fullres/fold_0"
if validation_complete(folder, CONTRACT["val"]):
    metrics = json.loads((folder / "validation/summary.json").read_text())
    print(json.dumps(subgroup_metrics(metrics, CONTRACT["val"]), indent=2))
    print("Yalnız 16 doğrulama vakası — ön sonuç (charm etiketlerine göre):")
    print(json.dumps({"foreground_mean": metrics.get("foreground_mean"), "per_label_mean": metrics.get("mean")}, indent=2))
else:
    print("Tamamlanmış 100 epoch doğrulama özeti henüz yok.")
print("Ara verirken checkpoint/logların Drive'da olduğunu kontrol et; kullanılmayan GPU oturumunu kapat.")


Sources: E2 workflow `tools/e2_training`; Dataset003 handoff reviewed at upstream `bfbb98d`; generator `tools/e3_training/build_e3_notebook.py`.
